In [37]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.formula.api import ols


In [38]:
df = pd.read_csv('bank.csv')

In [39]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [40]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        11162 non-null  int64 
 1   job        11162 non-null  object
 2   marital    11162 non-null  object
 3   education  11162 non-null  object
 4   default    11162 non-null  object
 5   balance    11162 non-null  int64 
 6   housing    11162 non-null  object
 7   loan       11162 non-null  object
 8   contact    11162 non-null  object
 9   day        11162 non-null  int64 
 10  month      11162 non-null  object
 11  duration   11162 non-null  int64 
 12  campaign   11162 non-null  int64 
 13  pdays      11162 non-null  int64 
 14  previous   11162 non-null  int64 
 15  poutcome   11162 non-null  object
 16  deposit    11162 non-null  object
dtypes: int64(7), object(10)
memory usage: 1.4+ MB


In [41]:
df.shape

(11162, 17)

In [42]:
df.columns

Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome', 'deposit'],
      dtype='object')

In [43]:
df['deposit'].unique()

array(['yes', 'no'], dtype=object)

In [44]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
deposit      0
dtype: int64

In [45]:
df.duplicated().sum()

np.int64(0)

In [46]:
categorical_cols = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
numerical_cols = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
target = 'deposit'

print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)
print("Target:", target)

Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Target: deposit


In [47]:
# Chi-Square Test for categorical features
from scipy.stats import chi2_contingency

print("=" * 70)
print("CHI-SQUARE TEST (Categorical vs Defaulted)")
print("=" * 70)

chi2_results = {}

for col in categorical_cols:
    table = pd.crosstab(df[col], df[target])
    chi2, p, dof, expected = chi2_contingency(table)

    chi2_results[col] = {'Chi2': chi2, 'p-value': p, 'dof': dof}

    print(f"\n{col} vs {target}")
    display(f"  Chi2: {chi2:.4f}, p-value: {p:.4f}, dof: {dof}")

    if p < 0.05:
        print(f"  ✅ Significant association! (p < 0.05)")
    else:
        print(f"  ❌ No significant association (p >= 0.05)")

# Summary
chi2_df = pd.DataFrame(chi2_results).T
chi2_df = chi2_df.sort_values('p-value')
print("\n" + "=" * 70)
print("CHI-SQUARE SUMMARY")
print("=" * 70)

pd.set_option('display.max_rows', None)
display(chi2_df)



CHI-SQUARE TEST (Categorical vs Defaulted)

job vs deposit


'  Chi2: 378.0753, p-value: 0.0000, dof: 11'

  ✅ Significant association! (p < 0.05)

marital vs deposit


'  Chi2: 109.5834, p-value: 0.0000, dof: 2'

  ✅ Significant association! (p < 0.05)

education vs deposit


...

'  Chi2: 1046.7745, p-value: 0.0000, dof: 11'

  ✅ Significant association! (p < 0.05)

poutcome vs deposit


'  Chi2: 1004.6358, p-value: 0.0000, dof: 3'

  ✅ Significant association! (p < 0.05)

CHI-SQUARE SUMMARY


,Chi2,p-value,dof
month,1046.774503,1.642083e-217,11.0
poutcome,1004.635780,1.776185e-217,3.0
contact,736.686680,1.072803e-160,2.0
housing,463.189241,9.724394e-103,1.0
job,378.075256,2.741690e-74,11.0
loan,135.832171,2.171287e-31,1.0
education,122.770090,1.953419e-26,3.0
marital,109.583356,1.600577e-24,2.0
default,17.808574,2.442800e-05,1.0


In [48]:
print("\n" + "=" * 70)
print("ANOVA TEST (Numerical vs Churn)")
print("=" * 70)

anova_results = {}

for col in numerical_cols:
    formula = f'{col} ~ C({target})'
    model = ols(formula, df).fit()
    aov_table = sm.stats.anova_lm(model, typ=2)

    f_value = aov_table['F'].iloc[0]
    p_value = aov_table['PR(>F)'].iloc[0]

    anova_results[col] = {'F-value': f_value, 'p-value': p_value}

    print(f"\n{col} vs {target}")
    display(f"  F-value: {f_value:.4f}, p-value: {p_value:.4f}")

    if p_value < 0.05:
        print(f"  ✅ Significant difference! (p < 0.05)")
    else:
        print(f"  ❌ No significant difference (p >= 0.05)")

# Summary
anova_df = pd.DataFrame(anova_results).T
anova_df = anova_df.sort_values('p-value')
print("\n" + "=" * 70)
print("ANOVA SUMMARY")
print("=" * 70)
display(anova_df)


ANOVA TEST (Numerical vs Churn)

age vs deposit


'  F-value: 13.6103, p-value: 0.0002'

  ✅ Significant difference! (p < 0.05)

balance vs deposit


'  F-value: 73.9401, p-value: 0.0000'

  ✅ Significant difference! (p < 0.05)

day vs deposit


...

'  F-value: 262.4922, p-value: 0.0000'

  ✅ Significant difference! (p < 0.05)

previous vs deposit


'  F-value: 222.6772, p-value: 0.0000'

  ✅ Significant difference! (p < 0.05)

ANOVA SUMMARY


,F-value,p-value
duration,2864.172247,0.000000e+00
pdays,262.492195,2.271607e-58
previous,222.677159,7.125338e-50
campaign,186.129893,4.831324e-42
balance,73.940086,9.126568e-18
day,35.518905,2.602203e-09
age,13.610293,2.260180e-04


In [49]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [50]:
df = df.drop(columns=['duration', 'default', 'age', 'day', 'balance'])

In [51]:
print("Categorical:", categorical_cols)
print("Numerical:", numerical_cols)
print("Target:", target)

Categorical: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']
Numerical: ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Target: deposit


In [52]:
from sklearn.preprocessing import OrdinalEncoder
le=OrdinalEncoder()
df[['job']]=le.fit_transform(df[['job']])
df[['marital']]=le.fit_transform(df[['marital']])
df[['education']]=le.fit_transform(df[['education']])
# df[['default']]=le.fit_transform(df[['default']])
df[['housing']]=le.fit_transform(df[['housing']])
df[['loan']]=le.fit_transform(df[['loan']])
df[['contact']]=le.fit_transform(df[['contact']])
df[['month']]=le.fit_transform(df[['month']])
df[['poutcome']]=le.fit_transform(df[['poutcome']])
# df[['age']]=le.fit_transform(df[['age']])
# df[['balance']]=le.fit_transform(df[['balance']])
# df[['day']]=le.fit_transform(df[['day']])
# df[['duration']]=le.fit_transform(df[['duration']])
df[['campaign']]=le.fit_transform(df[['campaign']])
df[['pdays']]=le.fit_transform(df[['pdays']])
df[['previous']]=le.fit_transform(df[['previous']])

In [53]:
from sklearn.preprocessing import LabelEncoder
o = LabelEncoder()
df['deposit'] = o.fit_transform(df['deposit'])

In [54]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 12 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   job        11162 non-null  float64
 1   marital    11162 non-null  float64
 2   education  11162 non-null  float64
 3   housing    11162 non-null  float64
 4   loan       11162 non-null  float64
 5   contact    11162 non-null  float64
 6   month      11162 non-null  float64
 7   campaign   11162 non-null  float64
 8   pdays      11162 non-null  float64
 9   previous   11162 non-null  float64
 10  poutcome   11162 non-null  float64
 11  deposit    11162 non-null  int64  
dtypes: float64(11), int64(1)
memory usage: 1.0 MB


In [55]:
df['deposit'].value_counts()

deposit
0    5873
1    5289
Name: count, dtype: int64

In [56]:
df.head()

,job,marital,education,housing,loan,contact,month,campaign,pdays,previous,poutcome,deposit
0,0.0,1.0,1.0,1.0,0.0,2.0,8.0,0.0,0.0,0.0,3.0,1
1,0.0,1.0,1.0,0.0,0.0,2.0,8.0,0.0,0.0,0.0,3.0,1
2,9.0,1.0,1.0,1.0,0.0,2.0,8.0,0.0,0.0,0.0,3.0,1
3,7.0,1.0,1.0,1.0,0.0,2.0,8.0,0.0,0.0,0.0,3.0,1
4,0.0,1.0,2.0,0.0,0.0,2.0,8.0,1.0,0.0,0.0,3.0,1


In [57]:
x = df.drop('deposit', axis=1)
y = df['deposit']

from sklearn.model_selection import train_test_split

# Train-Test Split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)
print("X_train shape:", x_train.shape)
print("X_test shape:", x_test.shape)

X_train shape: (8929, 11)
X_test shape: (2233, 11)


In [58]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [59]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=4,
    class_weight='balanced'
    )
rf.fit(x_train_scaled, y_train)

rf_train_pred = rf.predict(x_train_scaled)
rf_test_pred = rf.predict(x_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print("=" * 60)
print("Random Forest Evaluation")
print("=" * 60)
print(f"Train Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:  {rf_test_acc:.4f}")

print(f"Gap:            {abs(rf_train_acc - rf_test_acc):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_train, rf_train_pred))
print(classification_report(y_test, rf_test_pred))


Random Forest Evaluation
Train Accuracy: 0.6999
Test Accuracy:  0.7004
Gap:            0.0005

Test Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.78      0.73      4707
           1       0.72      0.61      0.66      4222

    accuracy                           0.70      8929
   macro avg       0.70      0.70      0.69      8929
weighted avg       0.70      0.70      0.70      8929

              precision    recall  f1-score   support

           0       0.69      0.78      0.73      1166
           1       0.72      0.61      0.66      1067

    accuracy                           0.70      2233
   macro avg       0.70      0.70      0.70      2233
weighted avg       0.70      0.70      0.70      2233



In [60]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(
    max_depth=5,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=4
)

dt.fit(x_train_scaled, y_train)


rf_train_pred = rf.predict(x_train_scaled)
rf_test_pred = rf.predict(x_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print("=" * 60)
print("Random Forest Evaluation")
print("=" * 60)
print(f"Train Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:  {rf_test_acc:.4f}")

print(f"Gap:            {abs(rf_train_acc - rf_test_acc):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_train, rf_train_pred))
print(classification_report(y_test, rf_test_pred))


Random Forest Evaluation
Train Accuracy: 0.6999
Test Accuracy:  0.7004
Gap:            0.0005

Test Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.78      0.73      4707
           1       0.72      0.61      0.66      4222

    accuracy                           0.70      8929
   macro avg       0.70      0.70      0.69      8929
weighted avg       0.70      0.70      0.70      8929

              precision    recall  f1-score   support

           0       0.69      0.78      0.73      1166
           1       0.72      0.61      0.66      1067

    accuracy                           0.70      2233
   macro avg       0.70      0.70      0.70      2233
weighted avg       0.70      0.70      0.70      2233



In [61]:
from sklearn.linear_model import LogisticRegression
lr = LogisticRegression()


dt.fit(x_train_scaled, y_train)


rf_train_pred = rf.predict(x_train_scaled)
rf_test_pred = rf.predict(x_test_scaled)

rf_train_acc = accuracy_score(y_train, rf_train_pred)
rf_test_acc = accuracy_score(y_test, rf_test_pred)

print("=" * 60)
print("Random Forest Evaluation")
print("=" * 60)
print(f"Train Accuracy: {rf_train_acc:.4f}")
print(f"Test Accuracy:  {rf_test_acc:.4f}")

print(f"Gap:            {abs(rf_train_acc - rf_test_acc):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_train, rf_train_pred))
print(classification_report(y_test, rf_test_pred))


Random Forest Evaluation
Train Accuracy: 0.6999
Test Accuracy:  0.7004
Gap:            0.0005

Test Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.78      0.73      4707
           1       0.72      0.61      0.66      4222

    accuracy                           0.70      8929
   macro avg       0.70      0.70      0.69      8929
weighted avg       0.70      0.70      0.70      8929

              precision    recall  f1-score   support

           0       0.69      0.78      0.73      1166
           1       0.72      0.61      0.66      1067

    accuracy                           0.70      2233
   macro avg       0.70      0.70      0.70      2233
weighted avg       0.70      0.70      0.70      2233



In [62]:
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier
ada = AdaBoostClassifier(
    n_estimators=1000,
    learning_rate=1.0,
    random_state=42
)

ada.fit(x_train_scaled, y_train)
ada_train_pred = ada.predict(x_train_scaled)
ada_test_pred = ada.predict(x_test_scaled )

ada_train_acc = accuracy_score(y_train, ada_train_pred)
ada_test_acc = accuracy_score(y_test, ada_test_pred)

print("=" * 60)
print("AdaBoost Evaluation")
print("=" * 60)
print(f"Train Accuracy: {ada_train_acc:.4f}")
print(f"Test Accuracy:  {ada_test_acc:.4f}")
print(f"Gap:            {abs(ada_train_acc - ada_test_acc):.4f}")
print("\nTest Classification Report:")
print(classification_report(y_train, ada_train_pred))
print(classification_report(y_test, ada_test_pred))

AdaBoost Evaluation
Train Accuracy: 0.7006
Test Accuracy:  0.7080
Gap:            0.0074

Test Classification Report:
              precision    recall  f1-score   support

           0       0.69      0.79      0.74      4707
           1       0.72      0.60      0.65      4222

    accuracy                           0.70      8929
   macro avg       0.70      0.70      0.70      8929
weighted avg       0.70      0.70      0.70      8929

              precision    recall  f1-score   support

           0       0.69      0.80      0.74      1166
           1       0.74      0.60      0.66      1067

    accuracy                           0.71      2233
   macro avg       0.71      0.70      0.70      2233
weighted avg       0.71      0.71      0.70      2233

